# Módulo 2 — Limpeza de Dados

Dados reais de cadastro de consumidores raramente chegam limpos. Este notebook trata os problemas mais comuns encontrados em sistemas de distribuição de energia elétrica.

## O que vamos ver

1. Identificar e tratar valores nulos
2. Detectar e remover duplicatas
3. Converter tipos de dados
4. Normalizar strings (nomes, endereços)
5. Validar CPF/CNPJ com regex
6. Criar um score de qualidade por registro

---

> **Por que isso importa?** Dados de cadastro com erros afetam diretamente o faturamento, a comunicação com o consumidor, o cálculo de indicadores (DEC/FEC) e o cumprimento de obrigações regulatórias com a ANEEL.

In [ ]:
import pandas as pd
import numpy as np
import re

# Carregar dados
df_original = pd.read_csv("dados/consumidores_simulado.csv")

# Trabalhar em cópia para preservar original
df = df_original.copy()

print(f"Registros carregados: {len(df)}")
print(f"Colunas: {list(df.columns)}")

## 1. Identificando Valores Nulos

In [ ]:
# Mapa completo de nulos
nulos_por_coluna = pd.DataFrame({
    "total_nulos": df.isnull().sum(),
    "pct_nulos": (df.isnull().sum() / len(df) * 100).round(1)
}).sort_values("total_nulos", ascending=False)

print("Nulos por coluna:")
nulos_por_coluna[nulos_por_coluna["total_nulos"] > 0]

In [ ]:
# Registros com pelo menos um campo nulo
registros_com_nulo = df[df.isnull().any(axis=1)]
print(f"Registros com pelo menos um campo nulo: {len(registros_com_nulo)}")
registros_com_nulo[["cod_consumidor", "nom_consumidor", "cpf_cnpj", "num_medidor", "dat_ultima_leitura"]]

In [ ]:
# Estratégias para tratar nulos

# 1. CPF/CNPJ nulo: marcar como 'NAO_INFORMADO' para rastreabilidade
df["cpf_cnpj"] = df["cpf_cnpj"].fillna("NAO_INFORMADO")

# 2. Medidor nulo: preencher com 0 (indica ausência de medidor)
# Mas primeiro vamos converter para string para manter consistência
df["num_medidor"] = df["num_medidor"].fillna(0).astype(int)

# 3. Data de última leitura nula: manter como NaT (será tratado na análise de qualidade)
df["dat_ultima_leitura"] = pd.to_datetime(df["dat_ultima_leitura"], errors="coerce")

print("Tratamento de nulos aplicado.")
print(f"Nulos restantes em cpf_cnpj: {df['cpf_cnpj'].isnull().sum()}")
print(f"Nulos restantes em num_medidor: {df['num_medidor'].isnull().sum()}")
print(f"Nulos em dat_ultima_leitura (NaT): {df['dat_ultima_leitura'].isnull().sum()}")

## 2. Detectando e Removendo Duplicatas

In [ ]:
# Duplicatas de cod_consumidor (chave primária)
duplicatas_cod = df[df.duplicated(subset=["cod_consumidor"], keep=False)]
print(f"Registros com cod_consumidor duplicado: {len(duplicatas_cod)}")
if len(duplicatas_cod) > 0:
    print()
    duplicatas_cod[["cod_consumidor", "nom_consumidor", "vlr_consumo_medio_kwh"]]

In [ ]:
# Duplicatas por CPF/CNPJ (mesmo documento, cods diferentes = possível duplicata de pessoa)
cpfs_validos = df[df["cpf_cnpj"] != "NAO_INFORMADO"]
duplicatas_cpf = cpfs_validos[
    cpfs_validos.duplicated(subset=["cpf_cnpj"], keep=False)
]

print(f"Registros com CPF/CNPJ duplicado: {len(duplicatas_cpf)}")
if len(duplicatas_cpf) > 0:
    print()
    print(duplicatas_cpf[["cod_consumidor", "nom_consumidor", "cpf_cnpj"]].to_string())

In [ ]:
# Remover duplicatas de cod_consumidor mantendo o primeiro registro
n_antes = len(df)
df = df.drop_duplicates(subset=["cod_consumidor"], keep="first")
n_depois = len(df)

print(f"Registros antes: {n_antes}")
print(f"Registros após deduplicação: {n_depois}")
print(f"Removidos: {n_antes - n_depois}")

## 3. Conversão de Tipos de Dados

In [ ]:
# Verificar tipos antes
print("Tipos ANTES da conversão:")
print(df[["cod_consumidor", "flg_ativo", "vlr_consumo_medio_kwh", "dat_ultima_leitura"]].dtypes)

# flg_ativo pode ter chegado como string "True"/"False"
if df["flg_ativo"].dtype == object:
    df["flg_ativo"] = df["flg_ativo"].map({"True": True, "False": False, True: True, False: False})

# cod_consumidor como int
df["cod_consumidor"] = df["cod_consumidor"].astype(int)

# vlr_consumo: garantir float
df["vlr_consumo_medio_kwh"] = pd.to_numeric(df["vlr_consumo_medio_kwh"], errors="coerce")

print("\nTipos APÓS a conversão:")
print(df[["cod_consumidor", "flg_ativo", "vlr_consumo_medio_kwh", "dat_ultima_leitura"]].dtypes)

## 4. Normalização de Strings

In [ ]:
# Verificar inconsistências nos nomes
print("Nomes com formatação inconsistente (amostras):")
nomes_inconsistentes = df[
    df["nom_consumidor"].str.isupper() | df["nom_consumidor"].str.islower()
]["nom_consumidor"].head(10)
for nome in nomes_inconsistentes:
    print(f"  '{nome}'")

In [ ]:
def normalizar_nome(nome: str) -> str:
    """
    Normaliza o nome do consumidor:
    - Remove espaços extras
    - Aplica Title Case
    - Mantém siglas conhecidas em maiúsculas
    """
    if pd.isna(nome):
        return nome
    
    # Remover espaços extras e aplicar title case
    nome_limpo = " ".join(nome.strip().split()).title()
    
    # Manter preposições em minúsculas (exceto início)
    preposicoes = ["Da", "De", "Do", "Das", "Dos", "E", "Em"]
    palavras = nome_limpo.split()
    resultado = []
    for i, palavra in enumerate(palavras):
        if i > 0 and palavra in preposicoes:
            resultado.append(palavra.lower())
        else:
            resultado.append(palavra)
    
    return " ".join(resultado)

def normalizar_cep(cep: str) -> str:
    """Normaliza CEP para formato XXXXX-XXX."""
    if pd.isna(cep):
        return cep
    digitos = re.sub(r'\D', '', str(cep))
    if len(digitos) == 8:
        return f"{digitos[:5]}-{digitos[5:]}"
    return str(cep)  # retorna original se não tiver 8 dígitos


# Aplicar normalizações
df["nom_consumidor"] = df["nom_consumidor"].apply(normalizar_nome)
df["num_cep"] = df["num_cep"].apply(normalizar_cep)

# Normalizar cod_uf para maiúsculas
df["cod_uf"] = df["cod_uf"].str.upper().str.strip()

# Normalizar cod_classe para maiúsculas
df["cod_classe_consumo"] = df["cod_classe_consumo"].str.upper().str.strip()

print("Normalizações aplicadas.")
df[["cod_consumidor", "nom_consumidor", "num_cep", "cod_uf"]].head(10)

## 5. Validação de CPF/CNPJ com Regex

In [ ]:
def classificar_documento(doc: str) -> str:
    """
    Classifica o documento em: CPF, CNPJ, NAO_INFORMADO, FORMATO_INVALIDO.
    Não valida dígitos verificadores — apenas o comprimento.
    """
    if pd.isna(doc) or str(doc).strip() in ["", "NAO_INFORMADO"]:
        return "NAO_INFORMADO"
    
    # Remover caracteres não numéricos
    digitos = re.sub(r'\D', '', str(doc))
    
    if len(digitos) == 11:
        # Verificar se não são todos iguais
        if digitos == digitos[0] * 11:
            return "CPF_INVALIDO_SEQUENCIA"
        return "CPF"
    elif len(digitos) == 14:
        if digitos == digitos[0] * 14:
            return "CNPJ_INVALIDO_SEQUENCIA"
        return "CNPJ"
    else:
        return f"FORMATO_INVALIDO_{len(digitos)}_DIGITOS"


def limpar_documento(doc: str) -> str:
    """Remove formatação do CPF/CNPJ, retorna só os dígitos."""
    if pd.isna(doc) or str(doc) == "NAO_INFORMADO":
        return doc
    return re.sub(r'\D', '', str(doc))


# Adicionar colunas de diagnóstico
df["tipo_documento"] = df["cpf_cnpj"].apply(classificar_documento)
df["cpf_cnpj_limpo"] = df["cpf_cnpj"].apply(limpar_documento)

# Resumo da qualidade dos documentos
resumo_docs = df["tipo_documento"].value_counts()
print("Resumo dos documentos:")
for tipo, qtd in resumo_docs.items():
    print(f"  {tipo}: {qtd}")

# Registros com problemas
problemas = df[~df["tipo_documento"].isin(["CPF", "CNPJ"])]
print(f"\nRegistros com problemas no documento: {len(problemas)}")
problemas[["cod_consumidor", "nom_consumidor", "cpf_cnpj", "tipo_documento"]]

## 6. Score de Qualidade por Registro

In [ ]:
def calcular_score_qualidade(row: pd.Series) -> float:
    """
    Calcula um score de qualidade de 0 a 1 para cada registro.
    Baseado em campos obrigatórios do cadastro de UC.
    """
    pontuacao = 0
    total_verificacoes = 0
    
    # 1. Documento (peso 2)
    total_verificacoes += 2
    if row["tipo_documento"] in ["CPF", "CNPJ"]:
        pontuacao += 2
    elif row["tipo_documento"] != "NAO_INFORMADO":
        pontuacao += 1  # tem algo, mas com problema
    
    # 2. Nome do consumidor (peso 1)
    total_verificacoes += 1
    if pd.notna(row["nom_consumidor"]) and len(str(row["nom_consumidor"]).strip()) > 3:
        pontuacao += 1
    
    # 3. Endereço — municipio + UF (peso 1)
    total_verificacoes += 1
    if pd.notna(row["nom_municipio"]) and pd.notna(row["cod_uf"]):
        pontuacao += 1
    
    # 4. CEP válido (peso 1)
    total_verificacoes += 1
    if pd.notna(row["num_cep"]) and re.match(r'^\d{5}-\d{3}$', str(row["num_cep"])):
        pontuacao += 1
    
    # 5. Medidor informado (peso 1)
    total_verificacoes += 1
    if pd.notna(row["num_medidor"]) and row["num_medidor"] != 0:
        pontuacao += 1
    
    # 6. Data de última leitura (peso 1)
    total_verificacoes += 1
    if pd.notna(row["dat_ultima_leitura"]):
        pontuacao += 1
    
    # 7. Consumo razoável (peso 1) — entre 0 e 500.000 kWh
    total_verificacoes += 1
    consumo = row["vlr_consumo_medio_kwh"]
    if pd.notna(consumo) and 0 <= consumo <= 500000:
        pontuacao += 1
    
    return round(pontuacao / total_verificacoes, 3)


# Aplicar score
df["score_qualidade"] = df.apply(calcular_score_qualidade, axis=1)

# Distribuição dos scores
print("Distribuição dos scores de qualidade:")
print(df["score_qualidade"].describe().round(3))

# Histograma simples
bins = [0, 0.3, 0.5, 0.7, 0.9, 1.0]
labels = ["Crítico", "Ruim", "Regular", "Bom", "Excelente"]
df["categoria_qualidade"] = pd.cut(df["score_qualidade"], bins=bins, labels=labels, right=True)

print("\nRegistros por categoria de qualidade:")
print(df["categoria_qualidade"].value_counts().sort_index())

In [ ]:
# Registros com score mais baixo — precisam de atenção prioritária
criticos = df[df["score_qualidade"] < 0.7].sort_values("score_qualidade")

print(f"Registros com score < 0.7 (necessitam revisão): {len(criticos)}")
print()
criticos[
    ["cod_consumidor", "nom_consumidor", "tipo_documento", "score_qualidade", "categoria_qualidade"]
].reset_index(drop=True)

In [ ]:
# Salvar DataFrame limpo
df.to_csv("dados/consumidores_limpo.csv", index=False)
print(f"DataFrame limpo salvo: {len(df)} registros")
print(f"Score médio de qualidade: {df['score_qualidade'].mean():.3f}")